---

## ✅ Setup Complete!

Your Colab environment is now configured with:
- ✅ PyTorch with CUDA support
- ✅ GPU detected and verified
- ✅ All dependencies installed
- ✅ Project structure created
- ✅ Google Drive mounted (for saving checkpoints)
- ✅ CUDA optimizations enabled

**Next Steps:**
1. Upload your project Python files
2. Upload training images
3. Run training command
4. Monitor with TensorBoard

**Estimated Training Times on Colab GPUs:**
- T4: ~1-2 hours for 30 epochs (2000 images)
- P100: ~45-60 minutes for 30 epochs
- V100/A100: ~30-40 minutes for 30 epochs

---

*Note: Colab free tier has usage limits. For longer training, consider Colab Pro or download checkpoints regularly to Google Drive.*

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Start TensorBoard (run this after training starts)
# %tensorboard --logdir runs/

print("📊 TensorBoard is ready!")
print("Uncomment the line above (%tensorboard --logdir runs/) after training starts")

## 📊 Monitor Training with TensorBoard (Optional)

Run this cell in a separate tab to monitor training progress in real-time.

## 📋 Ready to Train!

**Everything is now set up!** 

### Before starting training:

1. **Upload your project files** to this Colab environment:
   - `train.py`
   - `models/` folder (encoder.py, decoder.py, model.py)
   - `utils/` folder (if needed)
   - `losses.py`

2. **Upload training images** to `data/` folder or Google Drive

3. **Configure training parameters** based on your GPU:

```python
# T4 GPU (15GB):
python train.py --train_dir data/YOUR_IMAGES --num_epochs 30 --batch_size 16 --message_length 512 --image_size 128 --num_workers 2

# P100 GPU (16GB):
python train.py --train_dir data/YOUR_IMAGES --num_epochs 30 --batch_size 24 --message_length 512 --image_size 128 --num_workers 4

# V100 GPU (16GB):
python train.py --train_dir data/YOUR_IMAGES --num_epochs 50 --batch_size 32 --message_length 1024 --image_size 128 --num_workers 4
```

### To start training, run:

```bash
!python train.py --train_dir data/test_images --num_epochs 10 --batch_size 16 --message_length 512 --image_size 128 --save_freq 5
```

**Don't start training yet!** Make sure all your files are uploaded first.

In [ ]:
import sys
import torch
import torchvision
import PIL
import numpy as np
import matplotlib
from pathlib import Path

print("=" * 60)
print("INSTALLATION VERIFICATION")
print("=" * 60)

# Check Python packages
packages = {
    'PyTorch': torch.__version__,
    'TorchVision': torchvision.__version__,
    'PIL (Pillow)': PIL.__version__,
    'NumPy': np.__version__,
    'Matplotlib': matplotlib.__version__,
}

print("\n📦 Installed Packages:")
for name, version in packages.items():
    print(f"   {name:15s}: {version}")

# Check CUDA
print(f"\n🎮 CUDA Available: {'✅ Yes' if torch.cuda.is_available() else '❌ No'}")
if torch.cuda.is_available():
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# Check directories
print("\n📁 Directory Structure:")
dirs_to_check = ['data/test_images', 'checkpoints', 'runs']
for dir_path in dirs_to_check:
    exists = Path(dir_path).exists()
    status = "✅" if exists else "❌"
    print(f"   {status} {dir_path}")
    
    if dir_path == 'data/test_images' and exists:
        num_images = len(list(Path(dir_path).glob('*.jpg')))
        print(f"      ({num_images} images found)")

# Check if Google Drive is mounted
gdrive_mounted = Path('/content/drive/MyDrive').exists()
print(f"\n💾 Google Drive: {'✅ Mounted' if gdrive_mounted else '❌ Not mounted'}")

print("\n" + "=" * 60)
print("SETUP STATUS")
print("=" * 60)

all_good = all([
    torch.cuda.is_available(),
    Path('data/test_images').exists(),
    Path('checkpoints').exists(),
])

if all_good:
    print("✅ All checks passed! Ready for training!")
    print("\n📝 Next Steps:")
    print("   1. Upload your project files (train.py, models/, utils/)")
    print("   2. Upload training images to data/ folder")
    print("   3. Run the training command (see next cell)")
else:
    print("⚠️ Some checks failed. Please review the output above.")

print("=" * 60)

## 🔟 Verify Installation and Setup

Final verification that everything is ready for training.

In [ ]:
import os
import torch

# Set environment variables for optimal GPU performance
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'  # Async CUDA operations
os.environ['TORCH_CUDNN_V8_API_ENABLED'] = '1'  # Enable cuDNN v8 API

# Set PyTorch to use TF32 for faster training on newer GPUs
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # Auto-tune for best performance
    
    print("✅ CUDA environment configured for optimal performance")
    print("   - Async CUDA operations: Enabled")
    print("   - cuDNN benchmark mode: Enabled")
    print("   - TF32 acceleration: Enabled")
else:
    print("⚠️ No GPU available - running on CPU")

## 9️⃣ Set Environment Variables and CUDA Configuration

Configure optimal settings for Colab GPU training.

In [ ]:
import torch
import time

print("=" * 60)
print("GPU PERFORMANCE TEST")
print("=" * 60)

if torch.cuda.is_available():
    device = torch.device("cuda")
    
    # Test tensor operations on GPU
    print("Running GPU performance test...")
    
    # Create large tensors
    size = 10000
    x = torch.randn(size, size, device=device)
    y = torch.randn(size, size, device=device)
    
    # Warm up
    _ = torch.matmul(x, y)
    
    # Time matrix multiplication on GPU
    torch.cuda.synchronize()
    start = time.time()
    z = torch.matmul(x, y)
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    
    print(f"✅ GPU matrix multiplication ({size}x{size}): {gpu_time:.4f} seconds")
    print(f"   Performance: {(2 * size**3) / gpu_time / 1e9:.2f} GFLOPS")
    
    # Compare with CPU
    x_cpu = x.cpu()
    y_cpu = y.cpu()
    start = time.time()
    z_cpu = torch.matmul(x_cpu, y_cpu)
    cpu_time = time.time() - start
    
    print(f"   CPU matrix multiplication: {cpu_time:.4f} seconds")
    print(f"   GPU Speedup: {cpu_time / gpu_time:.1f}x faster")
    
    print("\n✅ GPU is working correctly and ready for training!")
else:
    print("❌ GPU not available - please enable GPU in runtime settings")

print("=" * 60)

## 8️⃣ Test GPU with PyTorch

Let's verify that PyTorch can actually use the GPU for computation.

In [ ]:
from PIL import Image
import numpy as np
from pathlib import Path

# Create a small test dataset
output_dir = Path('data/test_images')
output_dir.mkdir(parents=True, exist_ok=True)

num_images = 100  # Small dataset for quick testing
image_size = 128

print(f"Creating {num_images} synthetic test images...")
for i in range(num_images):
    # Create random RGB image
    img_array = np.random.randint(0, 256, (image_size, image_size, 3), dtype=np.uint8)
    img = Image.fromarray(img_array)
    img.save(output_dir / f'img_{i:04d}.jpg', 'JPEG', quality=95)
    
    if (i + 1) % 20 == 0:
        print(f"  Created {i + 1}/{num_images} images")

print(f"\n✅ Created {num_images} test images in {output_dir}")
print("\n📝 For real training, replace this with actual images:")
print("   Upload images to /content/drive/MyDrive/Steganography_Training/data/")

## 7️⃣ Create Sample Training Data

For testing, let's create a small dataset of synthetic images. 

**Note**: For real training, upload actual images to Google Drive.

In [ ]:
# Uncomment the lines below if cloning from GitHub
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO

# Or if you have the code in Google Drive:
# !cp -r /content/drive/MyDrive/Steganography_Training/* /content/
print("⚠️ Make sure your project files are available before proceeding!")

### Option B: Clone from GitHub (If Available)

Uncomment and run the cell below if your code is on GitHub:

In [ ]:
# Option A: If you've uploaded files to Google Drive
# Just change the working directory
import os
os.chdir('/content')

# Create local project structure
!mkdir -p models utils evaluation attacks defense
!mkdir -p checkpoints runs data

print("✅ Project directory structure created!")
print("\n📝 Next steps:")
print("1. Upload your Python files (train.py, models/, utils/, etc.) to:")
print("   /content/drive/MyDrive/Steganography_Training/")
print("\n2. Or continue below to clone from GitHub if available")

## 6️⃣ Setup Project Code

You have two options:

**Option A**: Upload your project files to Google Drive at `/content/drive/MyDrive/Steganography_Training/`

**Option B**: Clone from GitHub (if you've pushed your code to a repository)

For now, we'll create the project structure and copy the necessary files.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create project directory in Google Drive
project_dir = '/content/drive/MyDrive/Steganography_Training'
os.makedirs(project_dir, exist_ok=True)
os.makedirs(f'{project_dir}/checkpoints', exist_ok=True)
os.makedirs(f'{project_dir}/runs', exist_ok=True)
os.makedirs(f'{project_dir}/data', exist_ok=True)

print(f"✅ Google Drive mounted successfully!")
print(f"📁 Project directory: {project_dir}")
print(f"   Checkpoints will be saved to: {project_dir}/checkpoints")
print(f"   TensorBoard logs: {project_dir}/runs")
print(f"   Training data: {project_dir}/data")

## 5️⃣ Mount Google Drive (Optional but Recommended)

Mount Google Drive to save checkpoints and training logs permanently.

In [ ]:
import torch

print("=" * 60)
print("GPU DETECTION")
print("=" * 60)

if torch.cuda.is_available():
    print("✅ CUDA is available!")
    print(f"   GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"   Number of GPUs: {torch.cuda.device_count()}")
    
    # Get GPU memory
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"   Total GPU Memory: {gpu_memory:.2f} GB")
    
    # Current memory usage
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    print(f"   Allocated Memory: {allocated:.2f} GB")
    print(f"   Reserved Memory: {reserved:.2f} GB")
    print(f"   Free Memory: {gpu_memory - reserved:.2f} GB")
    
    # Recommended batch sizes based on GPU
    gpu_name = torch.cuda.get_device_name(0).lower()
    if 't4' in gpu_name:
        print("\n📊 Detected T4 GPU")
        print("   Recommended: --batch_size 16 --image_size 128")
    elif 'p100' in gpu_name:
        print("\n📊 Detected P100 GPU")
        print("   Recommended: --batch_size 24 --image_size 128")
    elif 'v100' in gpu_name:
        print("\n📊 Detected V100 GPU")
        print("   Recommended: --batch_size 32 --image_size 128")
    elif 'a100' in gpu_name:
        print("\n📊 Detected A100 GPU")
        print("   Recommended: --batch_size 48 --image_size 256")
    
else:
    print("❌ CUDA is NOT available!")
    print("⚠️ Make sure to enable GPU in Colab:")
    print("   Runtime → Change runtime type → Hardware accelerator → GPU")
    print("\n⚠️ Training without GPU will be extremely slow!")

print("=" * 60)

## 4️⃣ Verify GPU Availability

Let's check if GPU is available and get detailed information about it.

In [ ]:
!pip install pillow numpy matplotlib scikit-image tensorboard einops opencv-python tqdm

## 3️⃣ Install Additional Dependencies

Installing required libraries for image processing, visualization, and monitoring.

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

## 2️⃣ Install PyTorch with CUDA Support

Installing PyTorch with CUDA 11.8 for GPU acceleration.

In [ ]:
import sys
import platform
import os

print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)
print(f"Python Version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")
print(f"CPU Cores: {os.cpu_count()}")
print("=" * 60)

## 1️⃣ Check Python and System Information

First, let's verify the Python version and system details.

# 🚀 Deep Learning Steganography Training on Google Colab

This notebook is optimized for training on **Google Colab with GPU** (T4, P100, V100, or A100).

## Features:
- ✅ Automatic GPU detection (T4/P100/V100/A100)
- ✅ CUDA-enabled PyTorch installation
- ✅ Google Drive integration for saving checkpoints
- ✅ Optimized for Colab's free GPU tier
- ✅ Progress monitoring with TensorBoard

## Expected Performance:
- **T4 GPU**: 30 epochs on 2000 images → ~1-1.5 hours
- **P100 GPU**: 30 epochs on 2000 images → ~45-60 minutes
- **V100/A100**: 30 epochs on 2000 images → ~30-40 minutes

---

**⚠️ Important**: Run all cells in order from top to bottom!